# 介紹
使用 pyautogui 來完成小恐龍的遊戲機器人。

小恐龍是在 chrome 中使用 chrome://dino 啟動。

邏輯：

不斷監視畫面
讀取某個位置的顏色
若此位置上出現 (83, 83, 83) 的顏色
跳起來

In [83]:
import time
import pyautogui as pyg

pyg.PAUSE = 0.0001
pyg.sleep(2)
pyg.press('space')

start_time = time.time()

while True:
  current_time = time.time() - start_time

  # 【關鍵修改】剛開始速度慢，偵測點設在很近的 160
  # 隨著時間越跑越久、速度變快，才慢慢把距離往右拉遠（最多加 40 像素）
  extra_lookahead = min(int(current_time / 4), 40)
  scan_x = 160 + extra_lookahead

  # 截取局部畫面 (左, 上, 寬, 高)
  im = pyg.screenshot(region=(scan_x, 370, 50, 80))

  obs = False
  width, height = im.size

  for x in range(width):
    for y in range(height):
      c = im.getpixel((x, y))
      if c[0] < 120 and c[1] < 120 and c[2] < 120:
        obs = True
        break
    if obs:
      break

  if obs:
    pyg.press('space')
    time.sleep(0.08)

KeyboardInterrupt: 

In [ ]:
import time
import pyautogui as pyg

pyg.PAUSE = 0.0001
pyg.sleep(2)
pyg.press('space')

start_time = time.time()

while True:
  current_time = time.time() - start_time

  # 動態拉遠偵測距離
  extra_lookahead = min(int(current_time / 4), 40)
  scan_x = 160 + extra_lookahead

  # 1. 檢查背景顏色判斷日夜模式
  bg_sample = pyg.screenshot(region=(scan_x, 340, 10, 10))
  bg_color = bg_sample.getpixel((5, 5))
  is_dark_mode = bg_color[0] < 100

  # 2. 截取前方障礙物偵測區 (Y 軸 280 到 450，高度 170)
  im = pyg.screenshot(region=(scan_x, 280, 60, 170))
  width, height = im.size

  obstacle_ys = []

  for x in range(width):
    for y in range(height):
      c = im.getpixel((x, y))

      if is_dark_mode:
        is_obstacle = c[0] > 150 and c[1] > 150 and c[2] > 150
      else:
        is_obstacle = c[0] < 120 and c[1] < 120 and c[2] < 120

      if is_obstacle:
        obstacle_ys.append(y)

  # 3. 簡化為二段式邏輯：只有跳躍與直走
  if obstacle_ys:
    max_y = max(obstacle_ys)  # 抓出障礙物的最低點
    absolute_max_y = 280 + max_y  # 換算成絕對螢幕 Y 座標

    # 設分界線在 405：
    if absolute_max_y < 405:
      # 障礙物位置較高（頭頂上方） -> 直接直走，不動！
      pass
    else:
      # 障礙物在下方（地面或接近地面） -> 直接跳躍（space）
      pyg.press('space')
      time.sleep(0.08)     